# CRISP-DM Phase 2 — Data Understanding | RecoFit Dataset Exploration

**Project:** ML4B SoSe 2026 — Gym Exercise Recognition from Apple Watch Sensor Data  
**Dataset:** RecoFit (Microsoft Research, Morris et al., CHI 2014)  
**Date:** 2026-05-15  

## Goals of this notebook
1. Load the RecoFit `.mat` file using `scipy.io.loadmat`
2. Extract the full list of exercise classes
3. Understand the data structure (subjects × exercises cell matrix)
4. Analyse class distribution and data coverage
5. Inspect raw accelerometer and gyroscope signals
6. Run a data quality check (NaNs, missing recordings)
7. Confirm 50 Hz sampling rate empirically

**Output:** Completed `docs/data_understanding/dataset_evaluation.md` and updated target classes in `docs/business_understanding/business_understanding.md`.

## 1. Imports

In [ ]:
import scipy.io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
print('All imports successful.')

## 2. Data Paths

Paths are resolved relative to the project root using `pathlib` — no hardcoded absolute paths.  
Works identically on WSL, macOS, and Windows for all team members.

In [ ]:
# Path is relative to project root — works on all OS and for all team members
PROJECT_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'recofit'
MAT_FILE_SINGLE = DATA_RAW / 'exercise_data.50.0000_singleonly.mat'
MAT_FILE_MULTI  = DATA_RAW / 'exercise_data.50.0000_multionly.mat'

print(f'Project root:              {PROJECT_ROOT}')
print(f'Looking for data at:       {DATA_RAW}')
print(f'Single-activity file exists: {MAT_FILE_SINGLE.exists()}')
print(f'Multi-activity file exists:  {MAT_FILE_MULTI.exists()}')

## 3. Load Data

The `.mat` file is ~2.5 GB — loading takes 1–2 minutes on a typical machine.  
`simplify_cells=True` converts MATLAB cell arrays into Python lists/dicts automatically.

In [ ]:
# Loading takes ~1-2 minutes for 2.5 GB file
print('Loading single-activity data (this takes 1-2 minutes)...')
exercise_data = scipy.io.loadmat(str(MAT_FILE_SINGLE), simplify_cells=True)
print('Done!')
print('Top-level keys:', list(exercise_data.keys()))

## 4. Extract Exercise Class Names

The `exerciseConstants.activities` field contains the full list of exercise class label strings.  
This is the definitive source for our target class selection.

In [ ]:
activities = exercise_data['exerciseConstants']['activities']
print(f'Total number of exercise classes: {len(activities)}')
print('\nAll exercise classes:')
for i, name in enumerate(activities):
    print(f'  {i:3d}: {name}')

## 5. Map RecoFit Classes to Target Classes

After running Cell 4 and seeing the full class list, fill in the mapping below.  
Use the exact RecoFit label strings as keys and our standardized snake_case names as values.

**After filling this mapping, update `docs/business_understanding/business_understanding.md` Section 3 with the confirmed final class list.**

In [ ]:
# TODO: Fill this mapping after seeing the full class list in Cell 4 above
# Map RecoFit exercise names to our standardized target class names
EXERCISE_MAPPING = {
    # 'RecoFit exercise name': 'our_class_name',
    # Example: 'Two-arm Dumbbell Curl (both arms, not alternating)': 'bicep_curl',
}
print('Exercise mapping defined:', EXERCISE_MAPPING)
print(f'Mapped classes: {len(EXERCISE_MAPPING)}')

## 6. Data Coverage: Recordings per Exercise Class

The `subject_data` matrix has shape (n_subjects × n_exercises).  
Each cell contains the recordings for one subject doing one exercise type.

In [ ]:
subject_data = exercise_data['subject_data']
n_subjects, n_exercises = subject_data.shape
print(f'Number of subjects:        {n_subjects}')
print(f'Number of exercise types:  {n_exercises}')

# Count how many subjects have at least one recording per exercise
recordings_per_exercise = []
for i, activity in enumerate(activities):
    count = sum(
        1 for s in range(n_subjects)
        if subject_data[s, i] is not None
        and len(subject_data[s, i]) > 0
    )
    recordings_per_exercise.append({'exercise': activity, 'n_subjects_with_data': count})

df_counts = pd.DataFrame(recordings_per_exercise).sort_values(
    'n_subjects_with_data', ascending=False
)
print('\nRecordings per exercise (sorted by coverage):')
print(df_counts.to_string(index=False))

## 7. Visualise Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(6, len(activities) * 0.35)))
plot_df = df_counts.sort_values('n_subjects_with_data')
ax.barh(plot_df['exercise'], plot_df['n_subjects_with_data'], color='steelblue')
ax.set_xlabel('Number of subjects with recordings')
ax.set_title('RecoFit: Subject coverage per exercise class')
ax.axvline(x=n_subjects * 0.5, color='orange', linestyle='--', label='50% subject threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Inspect One Example Recording

Inspect the raw data structure and confirm the 50 Hz sampling rate empirically.  
Update `example_exercise_idx` after seeing the class list in Cell 4.

In [ ]:
# Inspect one example: update indices after seeing class list
example_subject      = 52   # subject index
example_exercise_idx = 0    # exercise index — update after seeing class list in Cell 4

recording = subject_data[example_subject, example_exercise_idx][0]
accel = recording['data']['accelDataMatrix']  # shape: (n_samples, 4) — col0=time, col1-3=xyz
gyro  = recording['data']['gyroDataMatrix']   # shape: (n_samples, 4) — col0=time, col1-3=xyz

print(f'Exercise:             {activities[example_exercise_idx]}')
print(f'Subject index:        {example_subject}')
print(f'Accelerometer shape:  {accel.shape}  (rows=samples, cols=[t, ax, ay, az])')
print(f'Gyroscope shape:      {gyro.shape}   (rows=samples, cols=[t, gx, gy, gz])')
print(f'Duration:             {accel[-1, 0] - accel[0, 0]:.2f} seconds')

# Empirically verify sampling rate from timestamp differences
dt_mean = np.mean(np.diff(accel[:, 0]))
fs_empirical = 1.0 / dt_mean
print(f'Empirical sampling rate: {fs_empirical:.1f} Hz  (expected: 50 Hz)')

## 9. Plot Raw Accelerometer and Gyroscope Signals

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Accelerometer
t = accel[:, 0] - accel[0, 0]  # relative time from 0
for col, label in zip([1, 2, 3], ['ax', 'ay', 'az']):
    axes[0].plot(t, accel[:, col], label=label, linewidth=0.8)
axes[0].set_ylabel('Acceleration (g)')
axes[0].set_title(
    f'Accelerometer — Subject {example_subject}, Exercise: {activities[example_exercise_idx]}'
)
axes[0].legend()

# Gyroscope
t_g = gyro[:, 0] - gyro[0, 0]
for col, label in zip([1, 2, 3], ['gx', 'gy', 'gz']):
    axes[1].plot(t_g, gyro[:, col], label=label, linewidth=0.8)
axes[1].set_ylabel('Angular velocity (dps)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Gyroscope')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Data Quality Check

Scan all recordings for NaN values and count empty subject–exercise slots.

In [ ]:
print('=== Data Quality Report ===\n')
nan_count      = 0
empty_count    = 0
total_recordings = 0

for s in range(n_subjects):
    for e in range(n_exercises):
        cell = subject_data[s, e]
        if cell is None or len(cell) == 0:
            empty_count += 1
            continue
        recs = cell if isinstance(cell, (list, np.ndarray)) else [cell]
        for rec in recs:
            total_recordings += 1
            try:
                data = rec['data']['accelDataMatrix']
                if np.isnan(data).any():
                    nan_count += 1
            except (KeyError, TypeError):
                pass

print(f'Total subject-exercise slots:  {n_subjects * n_exercises}')
print(f'Empty slots (no recording):    {empty_count}')
print(f'Total recordings inspected:    {total_recordings}')
print(f'Recordings with NaN values:    {nan_count}')
print()
print(f'Data quality: {"✅ No NaN values found" if nan_count == 0 else f"⚠️  {nan_count} recordings contain NaN"}')

## 11. Summary Findings

> Fill in after running all cells above.

| Finding | Value |
|---------|-------|
| Total exercise classes in RecoFit | ??? |
| Classes matching our target list | ??? |
| Recommended final target classes | ??? |
| Sampling rate (confirmed empirically) | 50 Hz |
| Number of subjects | ??? |
| Total recordings | ??? |
| Data quality issues (NaNs) | ??? |

### Next Steps for Phase 3 (Data Preparation)
- [ ] Finalize `EXERCISE_MAPPING` in Cell 5 with confirmed RecoFit label strings
- [ ] Update `docs/business_understanding/business_understanding.md` Section 3 with confirmed classes
- [ ] Design sliding-window segmentation (window size, step size) based on signal inspection
- [ ] Implement resampling if Apple Watch sampling rate differs from 50 Hz
- [ ] Create `src/ml4b/data/loader.py` with `load_recofit()` function